In [28]:
import numpy as np
import pandas as pd


def cor1(x,y,W):


    n_spots=W.shape[0]

    dx=x.shape[1]
    dy=y.shape[1]


    x_centered=x-np.mean(x, axis=0)
    y_centered=y-np.mean(y, axis=0)

    x_norm=np.sqrt(np.sum(x_centered**2, axis=0))
    y_norm=np.sqrt(np.sum(y_centered**2, axis=0))



    x_centered=x_centered/x_norm
    y_centered=y_centered/y_norm



    cor=np.zeros((dx,dy))
    for xi in range(dx):
        for yi in range(dy):
            cor[xi,yi]=np.sum(x_centered[W[:,0],xi]*y_centered[W[:,1],yi])*n_spots/(2*len(W))


    return cor

def cor2(x,y,W):


    n_spots=W.shape[0]

    dx=x.shape[1]
    dy=y.shape[1]


    x_centered=x-np.mean(x, axis=0)
    y_centered=y-np.mean(y, axis=0)

    x_norm=np.sqrt(np.sum(x_centered**2, axis=0))
    y_norm=np.sqrt(np.sum(y_centered**2, axis=0))

    cor=np.zeros((dx,dy))
    for xi in range(dx):
        for yi in range(dy):
            for [i,j] in W:
                cor[xi,yi]=cor[xi,yi]+x_centered[i,xi]*y_centered[j, yi]
            cor[xi,yi]= cor[xi,yi]*n_spots/(2*len(W))
            cor[xi,yi]= cor[xi,yi]/(x_norm[xi]*y_norm[yi])

    return cor


def cor0(x, y, W):
    n_spots = W.shape[0]

    dx = x.shape[1]
    dy = y.shape[1]

    x_centered = x - np.mean(x, axis=0)
    y_centered = y - np.mean(y, axis=0)

    x_norm = np.sqrt(np.sum(x_centered ** 2, axis=0))
    y_norm = np.sqrt(np.sum(y_centered ** 2, axis=0))

    x_centered=x_centered/x_norm
    y_centered=y_centered/y_norm

    cor=x_centered[W[:,0],:].T.dot(y_centered[W[:,1],:])*n_spots/(2*len(W))
    return cor


In [29]:
points = np.array([[0, 0], [0, 1], [0, 2], [1, 0], [1, 1], [1, 2],

                   [2, 0], [2, 1], [2, 2]])

from scipy.spatial import Voronoi, voronoi_plot_2d

vor = Voronoi(points)
vor.ridge_points
W=vor.ridge_points
W=np.vstack((W, np.stack((W[:,1], W[:,0])).T))
x=np.random.randn(9,4)
y=np.random.randn(9,3)

In [30]:
cor2(x, y, W)

array([[ 0.24726953, -0.26111456, -0.25024923],
       [ 0.24328895, -0.30566078,  0.23380297],
       [ 0.14067673, -0.09543962, -0.38380205],
       [ 0.33721983, -0.12788058,  0.16172064]])

In [31]:
cor1(x, y, W)

array([[ 0.24726953, -0.26111456, -0.25024923],
       [ 0.24328895, -0.30566078,  0.23380297],
       [ 0.14067673, -0.09543962, -0.38380205],
       [ 0.33721983, -0.12788058,  0.16172064]])

In [32]:
cor0(x, y, W)

array([[ 0.24726953, -0.26111456, -0.25024923],
       [ 0.24328895, -0.30566078,  0.23380297],
       [ 0.14067673, -0.09543962, -0.38380205],
       [ 0.33721983, -0.12788058,  0.16172064]])

In [153]:
def sci(A,B, X=None, W=None):

    if W is None:
        vor = Voronoi(X)
        W=vor.ridge_points
        W=np.vstack((W, np.stack((W[:,1], W[:,0])).T))

    elif W.shape[0]==W.shape[1]:
        W=np.stack(np.where(W), axis=0).T

    n_spots=np.max(W)+1

    if not (A.shape[0]==n_spots and B.shape[0]==n_spots):
        print("dimensions dont line up, dim 0 should be n_spots")


    if type(A) is pd.core.frame.DataFrame:
        x=A.to_numpy()
        y=B.to_numpy()
    else:
        x=A
        y=B

    # if method=="pearsonr":
    #     pass
    # elif method=="spearmanr":
    #     x=rankdata(x, axis=0)
    #     y=rankdata(y, axis=0)
    #
    # else:
    #     print("Warning: unknown corr method, using pearsonr")


    x_centered = x - np.mean(x, axis=0)
    y_centered = y - np.mean(y, axis=0)

    x_norm = np.sqrt(np.sum(x_centered ** 2, axis=0))
    y_norm = np.sqrt(np.sum(y_centered ** 2, axis=0))

    x_centered=x_centered/x_norm
    y_centered=y_centered/y_norm

    cor=x_centered[W[:,0],:].T.dot(y_centered[W[:,1],:])*n_spots/(2*len(W))

    if type(A) is pd.core.frame.DataFrame:
        cor=pd.DataFrame(cor, index=A.columns, columns=B.columns)

    return cor


def sci_test(A,B, X=None, W=None, n_trials=100):
    if type(A) is pd.core.frame.DataFrame:
        x=A.to_numpy()
        y=B.to_numpy()
    else:
        x=A
        y=B

    sci_true=sci(x,y, X=X, W=W)

    count=np.ones(sci_true.shape)
    for i in range(n_trials):
        count=count+(np.abs(sci_true)<=np.abs(sci(x[np.random.permutation(x.shape[0]),:],y[np.random.permutation(y.shape[0]),:], X=X, W=W)))

    return count/(n_trials+1)

    # yperm=np.hstack([y[np.random.permutation(y.shape[0])] for i in range(n_trial-1)])
    # yperm=np.hstack([y,yperm])
    #
    # rankdata(-np.abs(sci(x,yperm, X=None, W=W)).reshape((4,n_trial,3)), axis=1)[:,0,:]/n_trial

In [154]:
X=np.random.randn(100,2)

x=x
y=y
sci_true=sci(x,y, W=W)




In [156]:
sci_test(x,y, X=None, W=W, n_trials=100)



array([[0.31683168, 0.22772277, 0.2970297 ],
       [0.32673267, 0.21782178, 0.34653465],
       [0.59405941, 0.67326733, 0.15841584],
       [0.14851485, 0.54455446, 0.5049505 ]])

In [157]:
sci(x,y, W=W)

array([[ 0.09272607, -0.09791796, -0.09384346],
       [ 0.09123336, -0.11462279,  0.08767611],
       [ 0.05275377, -0.03578986, -0.14392577],
       [ 0.12645743, -0.04795522,  0.06064524]])

In [174]:
X=np.random.rand(100,2)
from spatrafact_util import make_kernel
K=make_kernel(X, 10)
w=np.random.randn(20,3)
x=K.dot(w)
y=K.dot(np.random.randn(20,3))

In [184]:
from scipy.stats import pearsonr

pearsonr(x[:,2], y[:,2])

(-0.3531073267525485, 0.00031397914728965816)

In [179]:
sci(x,y,X=X)

array([[-0.11199162, -0.09006704,  0.15758075],
       [ 0.1800308 , -0.04322831, -0.03346586],
       [ 0.23087689,  0.17238072, -0.17321649]])

In [152]:
yperm=np.hstack([y[np.random.permutation(y.shape[0])] for i in range(n_trial-1)])
yperm=np.hstack([y,yperm])

rankdata(-np.abs(sci(x,yperm, X=None, W=W)).reshape((4,n_trial,3)), axis=1)[:,0,:]/n_trial

array([[0.222, 0.223, 0.216],
       [0.519, 0.436, 0.545],
       [0.657, 0.769, 0.211],
       [0.118, 0.586, 0.481]])

In [140]:
sci(xperm,y, X=None, W=W)

array([[ 0.09272607, -0.09791796, -0.09384346],
       [ 0.09123336, -0.11462279,  0.08767611],
       [ 0.05275377, -0.03578986, -0.14392577],
       ...,
       [ 0.00280853,  0.09058012,  0.0012578 ],
       [-0.08348304, -0.05325758,  0.08273257],
       [ 0.07705817,  0.0804559 ,  0.16426359]])

In [103]:
xperm

array([[-1.49020043e-01, -1.23641582e+00,  1.44349661e+00, ...,
        -2.63534772e-01, -2.91636215e-02,  3.12250948e-01],
       [ 4.41844501e-01, -2.63534772e-01, -2.91636215e-02, ...,
        -1.23641582e+00,  1.44349661e+00,  4.55936323e-01],
       [ 6.28450420e-02,  4.06250857e-02, -2.39099958e+00, ...,
         7.67921526e-01,  7.74320613e-01, -3.15854065e-01],
       ...,
       [-6.93916308e-01, -1.21156435e+00, -9.80628854e-01, ...,
         4.06250857e-02, -2.39099958e+00,  5.21011254e-01],
       [ 3.89684705e-01,  1.12970362e-01,  1.36049902e+00, ...,
         1.12970362e-01,  1.36049902e+00, -3.32146768e-02],
       [-1.69801346e-03, -9.80300226e-02, -1.67273156e+00, ...,
         1.01383999e+00, -4.18546600e-01, -4.58041425e-01]])

In [124]:
from scipy.stats import rankdata
xperm[:,0:4]=np.ones((9,4))*2


array([[0.01, 0.01, 0.11, 0.01],
       [0.01, 0.01, 0.1 , 0.01],
       [0.01, 0.01, 0.14, 0.01],
       [0.01, 0.01, 0.06, 0.01],
       [0.01, 0.01, 0.09, 0.01],
       [0.01, 0.01, 0.12, 0.01],
       [0.01, 0.01, 0.12, 0.01],
       [0.01, 0.01, 0.19, 0.01],
       [0.01, 0.01, 0.15, 0.01]])

In [59]:
x=pd.DataFrame(x, columns=["a", "b", "c", "d"])
y=pd.DataFrame(y, columns=["x", "y", "z"])
sci(x,y,W=W)
sci(x, y, X=points)

,x,y,z
a,0.092726,-0.097918,-0.093843
b,0.091233,-0.114623,0.087676
c,0.052754,-0.035790,-0.143926
d,0.126457,-0.047955,0.060645


In [54]:
X=pd.DataFrame(points, index=[str(i)+"adads" for i in range(9)], columns=["x", "y"])

In [168]:
points

array([[0, 0],
       [0, 1],
       [0, 2],
       [1, 0],
       [1, 1],
       [1, 2],
       [2, 0],
       [2, 1],
       [2, 2]])